# QAOA vs. classical for QKD placement (Step 3 stretch goal)

The track README suggests using QAOA to choose which links get QKD under a budget, and comparing it with a classical choice. This notebook runs that comparison honestly.

**Instance.** The top-valued substations on the Chattanooga-area grid. Each gets one designated path (fewest hops) to a control center, and every link on those paths becomes one qubit. A substation counts only if **every** hop on its path gets QKD, and the budget is exactly K links. This is a simplified version of the full planner: one path per substation, no trusted-node cap, no control-center rule. That simplification is what makes it fit on 10–12 qubits.

**Methods compared**
- Exact: brute force over every budget-respecting answer, the same answer the integer program finds.
- Greedy: best value per link.
- Random sampling: any answer, and answers within the budget.
- **Standard QAOA**: the budget is enforced by a penalty term.
- **XY-mixer QAOA**: starts in an equal mix of all exactly-K answers, and its mixer swaps links in and out without changing how many are on, so no penalty is needed. This is the standard improvement for budget constraints.

**Main measure:** the chance that one shot returns the optimal answer. Here the optimum is easy to find classically, so the question is how often QAOA lands on it.

In [ ]:
import sys, json, math
sys.path.insert(0, "network_design")
import numpy as np, matplotlib.pyplot as plt
from qaoa_study import (build_instance, brute_force, greedy, run_qaoa, run_qaoa_xy, summarize,
                        random_baseline, random_feasible_baseline, value, bits_of)
NET = "network_design/regions/chattanooga_r1.json"
results = {}
for M, K in ((9, 5), (11, 6)):
    inst = build_instance(NET, M=M, K=K); nq = inst["nq"]
    opt, opt_bits = brute_force(inst); g, _ = greedy(inst)
    row = {"budget answers": math.comb(nq, K), "greedy / optimal": round(g / opt, 3),
           "random (any)": random_baseline(inst, opt)["p_optimal"],
           "random (within budget)": random_feasible_baseline(inst, opt, 100)["p_optimal"]}
    for p in (1, 2, 3):
        a = run_qaoa(inst, reps=p); b = run_qaoa_xy(inst, reps=p)
        row[f"standard QAOA p={p}"] = summarize(inst, a["probs"], opt)["p_optimal"]
        row[f"XY QAOA p={p}"] = summarize(inst, b["probs"], opt)["p_optimal"]
        row[f"two-qubit gates (standard, p={p})"] = a["two_qubit_gates"]
    results[f"{nq} qubits (K={K})"] = row
    if nq == 10: inst10, opt10, circ10 = inst, opt, run_qaoa(inst, reps=1)["circuit"]
for k, row in results.items():
    print(k); [print(f"   {a}: {b if isinstance(b, (int, str)) else round(100 * b, 3) if 'p=' in a or 'random' in a else b}" + ("%" if ('p=' in a and 'gates' not in a) or 'random' in a else "")) for a, b in row.items()]

In [ ]:
labels = ["random (any)", "random (within budget)", "standard QAOA p=1", "standard QAOA p=3", "XY QAOA p=1", "XY QAOA p=3"]
fig, ax = plt.subplots(figsize=(9, 4)); w = 0.38
for i, (k, row) in enumerate(results.items()):
    ax.bar(np.arange(len(labels)) + (i - 0.5) * w, [100 * row[l] for l in labels], w, label=k)
ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, fontsize=8, rotation=15)
ax.set_ylabel("chance one shot is optimal (%)"); ax.set_yscale("log")
ax.set_title("QAOA vs. random sampling (exact solver and greedy: optimal every time, ~0.01 s)", fontsize=9.5); ax.legend()
plt.tight_layout(); plt.savefig("3e_qaoa.png", dpi=150); plt.show()

## Run the 10-qubit circuit through qBraid (free simulators)

Standard QAOA with p = 1 compiles to about 100 two-qubit gates. That's too deep for meaningful results on today's noisy hardware, so this runs only on qBraid's free simulators. Copy the job IDs into the run record.

In [ ]:
from qiskit import transpile
from qbraid.runtime import QbraidProvider
import warnings; warnings.filterwarnings("ignore", message="Circuit contains measurement gates")
qc = transpile(circ10, basis_gates=["cx", "rz", "sx", "x", "h"], optimization_level=1); qc.measure_all()
provider = QbraidProvider()
for dev_id, opts in [("qbraid:qbraid:sim:qir-sv", {}), ("ionq:ionq:sim:simulator", {"runtime_options": {"noise_model": "aria-1"}})]:
    job = provider.get_device(dev_id).run(qc, shots=1000, **opts)
    counts = job.result().data.get_counts()
    vals = []
    for bitstring, c in counts.items():
        b = bits_of(int(bitstring.replace(" ", ""), 2), inst10["nq"])
        vals += [value(inst10, b) / opt10 if sum(b) == inst10["K"] else None] * c
    feas = [v for v in vals if v is not None]
    print(f"{dev_id}: job {job.id} | within budget {100*len(feas)/len(vals):.0f}% | optimal {100*sum(v > 0.9999 for v in feas)/len(vals):.2f}% | best {max(feas):.3f}")

## What this shows

- **Classical wins at every size tested.** The exact solver and greedy find the optimum every time in about 0.01 s, and the full planner (`3c`) solves real-sized versions exactly in seconds.
- **Standard QAOA learns the budget but not the value.** The share of answers within budget rises from 25% to about 60–70%, yet its chance of returning the optimum is *below* random guessing within the budget. The penalty term dominates the landscape.
- **XY-mixer QAOA does better: 2–4× random within budget** (0.45–1.7% at 10 qubits, 0.21–0.29% at 12). It would still need tens to hundreds of shots to find what the classical solver returns directly.
- **Hardware is the other barrier.** About 100–450 two-qubit gates is far beyond what today's noisy machines run reliably.
- **So the design uses quantum where physics gives an advantage (BB84 key exchange and tap detection) and classical optimization for planning.** That is a measured result from this notebook, not an assumption.